In [ ]:
# ==============================================================================
# CELDA 1: CONFIGURACIÓN Y MONTAJE DE GOOGLE DRIVE (GASTRO-AI PIPELINE v8)
# ==============================================================================
import os
import sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.model_selection import GroupKFold, KFold
from google.colab import drive

# 1. Montar Google Drive
if os.path.exists('/content/drive'):
    print('✓ Google Drive montado correctamente.')
else:
    drive.mount('/content/drive')

# 2. Configurar rutas absolutas en Google Drive
BASE_DIR = '/content/drive/MyDrive/GastroAI'
DATASET_DIR = os.path.join(BASE_DIR, 'dataset')
SPLITS_DIR = os.path.join(BASE_DIR, 'splits')
REPORTS_DIR = os.path.join(BASE_DIR, 'reports')

for folder in [DATASET_DIR, SPLITS_DIR, REPORTS_DIR]:
    os.makedirs(folder, exist_ok=True)

print('✓ Rutas de trabajo configuradas:')
print(f'  • Datasets: {DATASET_DIR}')
print(f'  • Splits:   {SPLITS_DIR}')
print(f'  • Reportes: {REPORTS_DIR}')


In [ ]:
# ==============================================================================
# CELDA 2: CARGA Y UNIFICACIÓN DE ANOTACIONES MULTIETIQUETA (v8 CON ATROFIA)
# ==============================================================================
def load_unified_annotations_v8(dataset_dir):
    """
    Integra los 3 datasets (EGID completo con Atrofia, GastroHUN e HyperKvasir Upper GI)
    con unificación de la Cascada de Pelayo Correa en vectores binarios de 4 clases.
    """
    csv_path = os.path.join(dataset_dir, 'annotations.csv')
    
    available_dirs = {}
    if os.path.exists(dataset_dir):
        for d in os.listdir(dataset_dir):
            full_d = os.path.join(dataset_dir, d)
            if os.path.isdir(full_d):
                available_dirs[d.lower()] = full_d
                
    print('[INFO] Carpetas de datasets detectadas en /dataset/:', list(available_dirs.keys()))
    records = []
    
    # -------------------------------------------------------------------------
    # 1. EGID (11,766 imágenes totales: H. Pylori + Presencia de Atrofia)
    # -------------------------------------------------------------------------
    egid_keys = [k for k in available_dirs.keys() if 'egid' in k]
    if egid_keys:
        egid_dir = available_dirs[egid_keys[0]]
        print(f'\n[PROCESANDO] EGID desde: {egid_dir}')
        
        # Mapeo de metadatos con fallbacks de lectura
        meta_file = os.path.join(egid_dir, 'metadata.xlsx')
        egid_meta_map = {}
        if os.path.exists(meta_file):
            df_meta = None
            try:
                df_meta = pd.read_excel(meta_file, engine='openpyxl')
            except Exception:
                try:
                    df_meta = pd.read_csv(meta_file)
                except Exception:
                    try:
                        df_meta = pd.read_excel(meta_file, engine='xlrd')
                    except Exception:
                        df_meta = None
            if df_meta is not None:
                df_meta.columns = [str(c).strip().lower() for c in df_meta.columns]
                for _, r in df_meta.iterrows():
                    ik = str(r.get('image_id', r.get('image', r.get('filename', '')))).strip()
                    pk = str(r.get('patient_id', r.get('patient', 'PAT_EGID_UNK'))).strip()
                    if ik:
                        egid_meta_map[ik] = pk

        img_count_egid = 0
        for root, _, files in os.walk(egid_dir):
            root_lower = root.lower()
            for f in files:
                if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff')):
                    img_count_egid += 1
                    full_path = os.path.join(root, f)
                    parts = full_path.split(os.sep)
                    patient_id = egid_meta_map.get(f, f"PAT_EGID_{parts[-2]}" if len(parts) >= 2 else f"PAT_EGID_{img_count_egid % 229:03d}")
                    
                    hpyl = 1 if ('positive' in root_lower or 'h_pylori' in root_lower or '1.h_pylori' in root_lower) else 0
                    atrophy = 1 if ('atrophy' in root_lower or '2.presence_of_atrophy' in root_lower or 'atrofia' in root_lower) else 0
                    gastritis = 1 if ('gastritis' in root_lower or hpyl == 1) else 0
                    normal = 1 if (hpyl == 0 and atrophy == 0 and gastritis == 0) else 0
                    
                    records.append({
                        'image_id': f,
                        'image_path': full_path,
                        'patient_id': patient_id,
                        'dataset_source': 'EGID',
                        'normal_mucosa': normal,
                        'erythematous_gastritis': gastritis,
                        'gastric_atrophy': atrophy,
                        'h_pylori_signs': hpyl
                    })
        print(f'  ✓ EGID finalizado: {img_count_egid} imágenes procesadas (H. Pylori + Atrofia).')

    # -------------------------------------------------------------------------
    # 2. GASTROHUN (8,834 imágenes en 387 pacientes)
    # -------------------------------------------------------------------------
    gastro_keys = [k for k in available_dirs.keys() if 'gastro' in k and 'egid' not in k]
    if gastro_keys:
        gastrohun_dir = available_dirs[gastro_keys[0]]
        print(f'\n[PROCESANDO] GastroHUN desde: {gastrohun_dir}')
        img_count_gh = 0
        for root, _, files in os.walk(gastrohun_dir):
            for f in files:
                if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff')):
                    img_count_gh += 1
                    full_path = os.path.join(root, f)
                    rel_path = os.path.relpath(full_path, gastrohun_dir)
                    folder_part = rel_path.split(os.sep)[0]
                    patient_id = f"PAT_GH_{folder_part}"
                    
                    records.append({
                        'image_id': f,
                        'image_path': full_path,
                        'patient_id': patient_id,
                        'dataset_source': 'GastroHUN',
                        'normal_mucosa': 1,
                        'erythematous_gastritis': 0,
                        'gastric_atrophy': 0,
                        'h_pylori_signs': 0
                    })
        print(f'  ✓ GastroHUN finalizado: {img_count_gh} imágenes procesadas.')

    # -------------------------------------------------------------------------
    # 3. HYPERKVASIR (3,452 imágenes de estómago / Upper GI)
    # -------------------------------------------------------------------------
    hyper_keys = [k for k in available_dirs.keys() if 'hyper' in k or 'kvasir' in k]
    if hyper_keys:
        hyperkvasir_dir = available_dirs[hyper_keys[0]]
        print(f'\n[PROCESANDO] HyperKvasir (solo Upper GI) desde: {hyperkvasir_dir}')
        img_count_hk = 0
        for root, _, files in os.walk(hyperkvasir_dir):
            root_lower = root.lower()
            if any(k in root_lower for k in ['lower-gi-tract', 'colon', 'polyps', 'cecum']):
                continue
                
            for f in files:
                if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff')):
                    img_count_hk += 1
                    full_path = os.path.join(root, f)
                    img_stem = os.path.splitext(f)[0]
                    patient_id = f"PAT_HK_{img_stem[:8]}"
                    
                    is_normal = any(k in root_lower for k in ['pylorus', 'retroflex', 'z-line'])
                    is_gastritis = 'esophagitis' in root_lower
                    is_atrophy = 'barrett' in root_lower
                    
                    normal = 1 if (is_normal and not is_gastritis and not is_atrophy) else 0
                    gastritis = 1 if is_gastritis else 0
                    atrophy = 1 if is_atrophy else 0
                    
                    records.append({
                        'image_id': f,
                        'image_path': full_path,
                        'patient_id': patient_id,
                        'dataset_source': 'HyperKvasir',
                        'normal_mucosa': normal,
                        'erythematous_gastritis': gastritis,
                        'gastric_atrophy': atrophy,
                        'h_pylori_signs': 0
                    })
        print(f'  ✓ HyperKvasir finalizado: {img_count_hk} imágenes procesadas.')

    df = pd.DataFrame(records)
    
    # Aplicar Regla de Exclusión Mutua para mucosa normal
    has_pathology = (df['erythematous_gastritis'] == 1) | (df['gastric_atrophy'] == 1) | (df['h_pylori_signs'] == 1)
    df.loc[has_pathology, 'normal_mucosa'] = 0
    df.loc[~has_pathology & (df['normal_mucosa'] == 0), 'normal_mucosa'] = 1
    
    df.to_csv(csv_path, index=False)
    print(f'\n✓ Registro unificado creado con {len(df)} muestras en: {csv_path}')
    return df

df_dataset = load_unified_annotations_v8(DATASET_DIR)

print('\n' + '='*65)
print(' 📊 RESUMEN UNIFICADO DE REGISTROS POR DATASET')
print('='*65)
print(df_dataset['dataset_source'].value_counts())

print('\n--- CONTEO DE MUESTRAS POSITIVAS POR CLASE PATOLÓGICA ---')
print(df_dataset[['normal_mucosa', 'erythematous_gastritis', 'gastric_atrophy', 'h_pylori_signs']].sum())


In [ ]:
# ==============================================================================
# CELDA 3: DIVISIÓN AISLADA POR PACIENTES (GroupKFold 5-Fold Cross Validation)
# ==============================================================================
def generate_patient_isolated_splits(df, n_splits=5, save_dir=None):
    """
    Aísla las muestras por patient_id garantizando cero fuga de información (data leakage).
    """
    df = df.copy()
    df['fold'] = -1
    unique_patients = df['patient_id'].nunique()
    
    if unique_patients >= n_splits:
        print(f'✓ Aplicando GroupKFold con {unique_patients} pacientes únicos...')
        gkf = GroupKFold(n_splits=n_splits)
        groups = df['patient_id'].values
        for fold, (train_idx, val_idx) in enumerate(gkf.split(df, groups=groups)):
            df.loc[val_idx, 'fold'] = fold
    else:
        print(f'⚠️ Pacientes únicos ({unique_patients}) < splits ({n_splits}). Conmutando a KFold...')
        kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
        for fold, (train_idx, val_idx) in enumerate(kf.split(df)):
            df.loc[val_idx, 'fold'] = fold

    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        out_csv = os.path.join(save_dir, 'patient_splits_5fold.csv')
        df.to_csv(out_csv, index=False)
        print(f'✓ Archivo de particiones guardado en Google Drive: {out_csv}')
        
    return df

df_splits = generate_patient_isolated_splits(df_dataset, n_splits=5, save_dir=SPLITS_DIR)

print('\nDistribución de imágenes por Fold (0 a 4):')
print(df_splits['fold'].value_counts().sort_index())


In [ ]:
# ==============================================================================
# CELDA 4: CÁLCULO DEL TENSOR POS_WEIGHTS PARA BCEWithLogitsLoss
# ==============================================================================
def calculate_pos_weights(df, target_columns):
    """Calcula el vector de pesos pos_weight para la BCEWithLogitsLoss multietiqueta."""
    total_samples = len(df)
    pos_weights = []
    print('\n--- TENSOR POS_WEIGHTS CALCULADO PARA PERDIDA MULTIETIQUETA ---')
    for col in target_columns:
        positives = int(df[col].sum())
        negatives = total_samples - positives
        weight = negatives / max(positives, 1)
        pos_weights.append(weight)
        print(f'  • {col}: Positivos={positives} | Negativos={negatives} | pos_weight={weight:.2f}')
    return torch.tensor(pos_weights, dtype=torch.float32)

target_columns = ['normal_mucosa', 'erythematous_gastritis', 'gastric_atrophy', 'h_pylori_signs']
pos_weights_tensor = calculate_pos_weights(df_splits, target_columns)
print(f'\nTensor PyTorch pos_weight listo para el modelo: {pos_weights_tensor}')


In [ ]:
# ==============================================================================
# CELDA 5: DATASET Y DATALOADERS DE PYTORCH (EfficientNet-B1 240x240)
# ==============================================================================
class GastroAIDataset(Dataset):
    """Dataset PyTorch para imágenes gástricas multietiqueta."""
    def __init__(self, df, label_cols, transform=None):
        self.df = df
        self.label_cols = label_cols
        self.transform = transform
        
    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, idx):
        img_path = self.df.iloc[idx]['image_path']
        
        if os.path.exists(img_path):
            try:
                image = Image.open(img_path).convert('RGB')
            except Exception:
                image = Image.fromarray(np.uint8(np.random.rand(240, 240, 3) * 255))
        else:
            image = Image.fromarray(np.uint8(np.random.rand(240, 240, 3) * 255))
            
        labels = self.df.iloc[idx][self.label_cols].values.astype(np.float32)
        
        if self.transform:
            image = self.transform(image)
            
        return image, torch.tensor(labels)

def get_transforms():
    """Transformaciones de Estandarización y Augmentación para EfficientNet-B1."""
    train_transform = transforms.Compose([
        transforms.Resize((240, 240)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomRotation(degrees=15),
        transforms.ColorJitter(brightness=0.1, contrast=0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    val_transform = transforms.Compose([
        transforms.Resize((240, 240)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    return train_transform, val_transform

train_transform, val_transform = get_transforms()

# Separación de entrenamiento (Folds 1-4) y validación (Fold 0)
train_df = df_splits[df_splits['fold'] != 0].reset_index(drop=True)
val_df = df_splits[df_splits['fold'] == 0].reset_index(drop=True)

train_dataset = GastroAIDataset(train_df, target_columns, transform=train_transform)
val_dataset = GastroAIDataset(val_df, target_columns, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)

print('✓ DataLoaders de PyTorch creados correctamente:')
print(f'  • Entrenamiento (Folds 1-4): {len(train_dataset)} muestras')
print(f'  • Validación    (Fold 0):   {len(val_dataset)} muestras')
